# Day 4 — Embeddings Deep Dive + Semantic Search from Scratch
---
> **Phase:** 1 — LLM Foundations  
> **Status:** ✅ Complete

## 🧠 Key Concepts

### 1. Why Avg Word2Vec Fails for Sentence Search

**Avg Word2Vec approach:**
- Take all word vectors in a sentence
- Average them into one single vector

**The problem — static vectors:**
```
Sentence 1: "The bank by the river was flooded"
Sentence 2: "The bank refused my loan application"
```
- Word2Vec gives "bank" the SAME vector in both sentences
- Avg Word2Vec produces SIMILAR vectors for both sentences
- A user searching "river flooding" would get loan rejection results
- That's a broken search system

**The solution — contextual embeddings:**
```
Word2Vec          → one vector per word, static, no context
Avg Word2Vec      → one vector per sentence, but still static word vectors
Modern Embeddings → one vector per sentence, fully contextual
                    trained specifically for semantic similarity
```

Modern embedding models (`all-MiniLM-L6-v2`, `text-embedding-3-small`) produce vectors where:
- `"bank" near "river"` → vector pointing toward nature/geography domain
- `"bank" near "money"` → vector pointing toward finance domain
- The ENTIRE sentence context determines the vector

### 2. Cosine Similarity — The Math

Cosine similarity measures the **angle between two vectors** — not the distance between their tips.

```
cosine_similarity = cos(θ)

cos(0°)   = 1.0   → identical direction → same meaning
cos(90°)  = 0.0   → perpendicular → unrelated
cos(180°) = -1.0  → opposite direction → opposite meaning
```

**Formula:**
```
cosine_similarity(A, B) = (A · B) / (||A|| × ||B||)

A · B    = dot product (how much they point in same direction)
||A||    = magnitude/length of vector A
||B||    = magnitude/length of vector B
dividing by magnitudes = normalize so only direction matters, not length
```

**For a search system:** You want cosine similarity as close to 1.0 as possible.

### 3. The Complete Semantic Search Pipeline

```
Step 1 → Embed all documents ONCE → store as matrix (n_docs, dimensions)
Step 2 → Embed query at search time → vector (dimensions,)
Step 3 → Compute cosine similarity → query vector vs every doc vector
Step 4 → Filter by threshold → only return confident matches
Step 5 → Return top K results → sorted by similarity score
```

**This is the retrieval engine inside every RAG system.**
Vector databases (FAISS, ChromaDB, Pinecone) do exactly this — at scale.

### 4. Vectorized Computation vs Loops

**Loop approach — slow:**
```python
for i, doc_vector in enumerate(document_vectors):  # Python loop
    similarity = cosine_similarity(query_vector, doc_vector)
# With 1 million documents → 1 million Python iterations → very slow
```

**Vectorized approach — fast:**
```python
dot_products = np.dot(document_vectors, query_vector)
# (10, 384) · (384,) = (10,) — all 10 similarities computed simultaneously
# With 1 million documents → 1 operation in C under the hood → fast
```

**Shape intuition:**
```
document_vectors: (10, 384)  — 10 documents, 384 dimensions each
query_vector:     (384,)     — 1 query, 384 dimensions
dot_products:     (10,)      — 1 similarity score per document
```

### 5. Production Lessons

**Semantic Drift:**
- Model finds closest documents even when nothing is truly relevant
- Scores of 0.30-0.35 = weak signal, model is stretching
- Always use a threshold to filter weak matches

**Threshold Tuning:**
```
Too low  → returns irrelevant results confidently
Too high → returns "No results" even when something useful exists
Solution → tune empirically by testing many queries on your data
```

**Variable Scope:**
```python
def semantic_search():
    results = [...]   # lives INSIDE function only, destroyed after return
    return results    # sends VALUE out, not the variable

results = semantic_search()  # completely separate variable, catches VALUE
# Both can be named anything — names don't matter, values do
```

**Separation of Concerns:**
```
Function responsibility → find and filter results, return clean list
Caller responsibility   → check if list is empty, handle display
Never mix printing and data building in same operation
```

## 💻 Code

In [ ]:
# Cosine Similarity from Scratch
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sentence_a = "The bank by the river was flooded"
sentence_b = "The river overflowed its banks after heavy rain"
sentence_c = "The bank refused my loan application"

vector_a = model.encode(sentence_a)
vector_b = model.encode(sentence_b)
vector_c = model.encode(sentence_c)

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    magnitude = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return dot_product / magnitude

sim_ab = cosine_similarity(vector_a, vector_b)
sim_ac = cosine_similarity(vector_a, vector_c)

print(f"River vs River:   {sim_ab:.4f}")  # expected: high ~0.73
print(f"River vs Finance: {sim_ac:.4f}")  # expected: low  ~0.37
print()
if sim_ab > sim_ac:
    print("✓ Contextual embeddings working correctly")

In [ ]:
# Semantic Search — Vectorized Version
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

documents = [
    "The battery life on this laptop is incredible, lasts all day",
    "Screen resolution is disappointing, colors look washed out",
    "Keyboard feels premium and typing experience is excellent",
    "The laptop runs very hot during heavy tasks, fan is loud",
    "Lightweight and portable, perfect for travel and commuting",
    "RAM is soldered and cannot be upgraded, big disappointment",
    "Boot time is blazing fast, SSD performance is outstanding",
    "Webcam quality is terrible, grainy even in good lighting",
    "Build quality feels solid, no flex in the chassis at all",
    "Price is too high for the specs you get with this machine"
]

# Embed documents ONCE
document_vectors = model.encode(documents)
print(f"Document matrix shape: {document_vectors.shape}")  # (10, 384)

def semantic_search(query, top_k=3, threshold=0.3):
    query_vector = model.encode(query)
    
    # Vectorized similarity — (10, 384) · (384,) = (10,)
    dot_products = np.dot(document_vectors, query_vector)
    query_norm = np.linalg.norm(query_vector)
    doc_norms = np.linalg.norm(document_vectors, axis=1)
    similarities = dot_products / (doc_norms * query_norm)
    
    # Get top K indices sorted by score
    top_indices = np.argsort(similarities)[::-1][:top_k]
    
    # Filter by threshold — clean list, no None values
    matches = [
        (similarities[i], i, documents[i])
        for i in top_indices
        if similarities[i] >= threshold
    ]
    return matches

# Test queries
queries = [
    "How long does the battery last?",
    "Is the display good quality?",
    "Does it come with a free charger?"  # should return nothing
]

for query in queries:
    print(f"\nQuery: '{query}'")
    print("-" * 50)
    results = semantic_search(query)
    
    if not results:
        print("No relevant reviews found.")
    else:
        for score, idx, doc in results:
            print(f"  Score: {score:.4f} | {doc}")

## 🔬 Observations

| Query | Top Result | Score | Lesson |
|-------|-----------|-------|--------|
| "How long does the battery last?" | "battery life...lasts all day" | 0.5066 | Semantic match without exact keywords |
| "Is the display good quality?" | "Screen resolution is disappointing" | 0.4889 | Matched despite different framing |
| "How heavy is it to carry?" | "Lightweight and portable" | 0.5319 | Synonym understanding |
| "Does it come with free charger?" | No results | — | Threshold correctly filtered irrelevant |
| River vs River similarity | 0.7337 | — | Contextual embeddings work |
| River vs Finance similarity | 0.3680 | — | "bank" disambiguation correct |

## ✅ Day 4 Summary

```
✓ Why Avg Word2Vec fails for sentence-level search
✓ How modern contextual embeddings solve it
✓ Cosine similarity — math, not just concept
✓ Implemented cosine similarity from scratch with numpy
✓ Built semantic search engine from scratch
✓ Vectorized computation vs loops — why it matters at scale
✓ Shape intuition: (10, 384) · (384,) = (10,)
✓ Threshold filtering — what it is and how to tune it
✓ Semantic drift — real production limitation
✓ Variable scope in Python
✓ Separation of concerns — function vs caller responsibilities
✓ Full semantic search assignment built independently
```

## 🔜 Coming in Day 5 — Vector Databases (FAISS)

**The problem with current approach:**
- Documents stored as numpy array in memory
- Every search recomputes similarities from scratch
- Does not scale beyond thousands of documents

**What FAISS solves:**
- Persistent storage of vectors on disk
- Approximate nearest neighbor search — much faster at scale
- Used in production systems with millions of documents

**Pre-Day 5 Question:**  
With 1 million documents, computing cosine similarity against all of them for every query is expensive.  
How do you think a vector database makes this faster without checking every single document?

concept is **indexing**.